# Supplementary Figure S3: Condition Centroid Analysis

- Figure / panels: `FigS3`
- Inputs: result payloads for the selected datasets
- Outputs: `artifacts/paper_figures/supp/FigS3_CentroidAnalysis/*`
- Run order: optional; run after the main benchmark notebooks
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Supplementary


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import scripts.trishift.analysis.condition_centroid_vis as condition_centroid_vis
importlib.reload(condition_centroid_vis)


In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

MODELS = ['trishift', 'biolord', 'gears', 'genepert', 'scgpt']  # edit here
VARIANT_TAGS = {'trishift': 'nearest'}
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS3_CentroidAnalysis'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_frames = []
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Selected models:', MODELS)
for spec in DATASET_SPECS:
    for split_id in SPLIT_IDS:
        for model_name in MODELS:
            try:
                result = condition_centroid_vis.run_condition_centroid_visualization(model_name=model_name, dataset=spec['dataset_key'], split_id=split_id, variant_tag=VARIANT_TAGS.get(model_name, ''), feature_mode='deg', include_ctrl=True, plot_delta=True, out_root=OUT_ROOT / spec['dataset_key'] / f'{model_name}_split{split_id}')
            except Exception:
                continue
            df = result.summary_df.copy(); df['dataset_label'] = spec['dataset_label']; df['model_name'] = model_name; df['split_id'] = split_id; summary_frames.append(df)
summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary_df.to_csv(OUT_ROOT / 'figs3_summary_all.csv', index=False, encoding='utf-8-sig')
display(summary_df.head())
print(OUT_ROOT)
